# BBEH × CAPO (promptolution)

Runs CAPO prompt optimization on BBEH mini (460 examples, 23 tasks) using `gpt-oss-120b` via Groq.

**Prerequisites:** Add your Groq API key to Colab Secrets as `GROQ_API_KEY`.

In [ ]:
# Cell 1: Install dependencies
!pip install -q "promptolution[api]" datasets

In [ ]:
# Cell 2: Shared config — download or paste shared_config.py
%%writefile shared_config.py

"""Shared constants for BBEH competitor comparison notebooks."""

MODEL_ID = "gpt-oss-120b"
API_BASE = "https://api.groq.com/openai/v1"
HF_DATASET = "BBEH/bbeh"
SPLIT_SEED = 42
TRAIN_PER_TASK = 10
TEST_PER_TASK = 10


def exact_match(expected: str, predicted: str) -> bool:
    return expected.strip().lower() == predicted.strip().lower()


def load_and_split():
    import random
    from datasets import load_dataset

    ds = load_dataset(HF_DATASET)["train"]
    mini = ds.filter(lambda x: x["mini"] == 1)

    by_task: dict[str, list[dict]] = {}
    for ex in mini:
        task = ex["task"]
        by_task.setdefault(task, []).append({"input": ex["input"], "target": ex["target"]})

    train_by_task: dict[str, list[dict]] = {}
    test_by_task: dict[str, list[dict]] = {}

    for task, examples in sorted(by_task.items()):
        rng = random.Random(SPLIT_SEED)
        shuffled = examples.copy()
        rng.shuffle(shuffled)
        train_by_task[task] = shuffled[:TRAIN_PER_TASK]
        test_by_task[task] = shuffled[TRAIN_PER_TASK:TRAIN_PER_TASK + TEST_PER_TASK]

    return train_by_task, test_by_task


def export_results(method, per_task, config, optimized_prompts, output_path="results.json"):
    import json
    from datetime import datetime, timezone

    accuracies = [t["accuracy"] for t in per_task.values()]
    overall = sum(accuracies) / len(accuracies) if accuracies else 0.0

    result = {
        "method": method,
        "model": MODEL_ID,
        "dataset": "bbeh-mini",
        "split_seed": SPLIT_SEED,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "config": config,
        "per_task": per_task,
        "overall_accuracy": round(overall, 4),
        "optimized_prompts": optimized_prompts,
    }

    with open(output_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"Results written to {output_path}")
    print(f"Overall accuracy (macro-avg): {overall:.1%}")
    return result

In [ ]:
# Cell 3: Load dataset and API key
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("GROQ_API_KEY")

from shared_config import (
    MODEL_ID, API_BASE, SPLIT_SEED, load_and_split,
    exact_match, export_results,
)

train_by_task, test_by_task = load_and_split()
tasks = sorted(train_by_task.keys())
print(f"Loaded {len(tasks)} tasks, {sum(len(v) for v in train_by_task.values())} train, "
      f"{sum(len(v) for v in test_by_task.values())} test")

In [ ]:
# Cell 4: Run CAPO per task
import pandas as pd
import nest_asyncio
from promptolution.utils import ExperimentConfig
from promptolution.helpers import run_experiment

nest_asyncio.apply()

N_STEPS = 10

optimized_prompts: dict[str, str] = {}
capo_config = {
    "optimizer": "capo",
    "n_steps": N_STEPS,
    "model_id": MODEL_ID,
}

for i, task in enumerate(tasks):
    print(f"\n[{i+1}/{len(tasks)}] Optimizing: {task}")

    # Build DataFrame in promptolution format
    train_df = pd.DataFrame([
        {"x": ex["input"], "y": ex["target"]}
        for ex in train_by_task[task]
    ])

    # Auto-generate task description from the task name
    task_desc = (
        f"Solve the following '{task.replace('_', ' ')}' problem. "
        f"Provide only the final answer, nothing else."
    )

    config = ExperimentConfig(
        optimizer="capo",
        task_description=task_desc,
        n_steps=N_STEPS,
        n_subsamples=len(train_df),  # use all train examples
        api_url=API_BASE,
        model_id=MODEL_ID,
        api_key=os.environ["OPENAI_API_KEY"],
    )

    result_prompts = run_experiment(train_df, config)

    # Take best prompt from the returned population
    best_prompt = result_prompts[0] if isinstance(result_prompts, list) else str(result_prompts)
    optimized_prompts[task] = best_prompt
    print(f"  Best prompt: {best_prompt[:100]}...")

In [ ]:
# Cell 5: Evaluate optimized prompts on held-out test set
from openai import OpenAI

client = OpenAI(base_url=API_BASE, api_key=os.environ["OPENAI_API_KEY"])

per_task_results: dict[str, dict] = {}

for i, task in enumerate(tasks):
    print(f"\n[{i+1}/{len(tasks)}] Evaluating: {task}")
    prompt = optimized_prompts[task]
    test_examples = test_by_task[task]
    correct = 0

    for ex in test_examples:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": ex["input"]},
            ],
            temperature=0.0,
            max_tokens=512,
        )
        predicted = response.choices[0].message.content or ""
        if exact_match(ex["target"], predicted):
            correct += 1

    accuracy = correct / len(test_examples)
    per_task_results[task] = {"accuracy": round(accuracy, 4), "n_test": len(test_examples)}
    print(f"  {task}: {correct}/{len(test_examples)} = {accuracy:.0%}")

In [ ]:
# Cell 6: Export results
result = export_results(
    method="capo",
    per_task=per_task_results,
    config=capo_config,
    optimized_prompts=optimized_prompts,
    output_path="results_capo.json",
)

# Print summary table
print("\n" + "="*50)
print(f"{'Task':<25} {'Accuracy':>8}")
print("-"*50)
for task in sorted(per_task_results):
    acc = per_task_results[task]["accuracy"]
    print(f"{task:<25} {acc:>8.0%}")
print("-"*50)
print(f"{'MACRO AVERAGE':<25} {result['overall_accuracy']:>8.1%}")